In [1]:
# Create one row of original company-event features.
# Preserve source values, missingness, targets, and event metadata.

In [2]:
# Locate shared project configuration and reusable helpers.
import sys as _sys
from pathlib import Path as _Path

_UTILS_ROOT = next(
    path
    for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
    if (path / "notebook_utils.py").is_file()
)
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths
from project_config import (
    ORIGINAL_COLUMNS as original_columns,
    SIGNED_LOG_FIELDS,
)
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/event_aligned_company_data.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed

aligned_company = pd.read_csv(PROCESSED / "event_aligned_company_data.csv")
company_availability = pd.read_csv(
    PROCESSED / "company_availability.csv", usecols=["company_id", "bankruptcy_date"]
)
required = [
    "event_id",
    "company_id",
    "prediction_date",
    "target",
    "prediction_year",
    "outcome_end_date",
    "bloomberg_sector",
    "bloomberg_industry",
] + original_columns
missing_columns = sorted(set(required) - set(aligned_company.columns))
if missing_columns:
    raise ValueError(f"Required aligned columns are missing: {missing_columns}")
if aligned_company.empty:
    raise ValueError("No aligned events; inspect notebook 07")
if company_availability["company_id"].duplicated().any():
    raise ValueError("Company availability must be unique by company_id")
if aligned_company[["event_id", "company_id", "prediction_date"]].isna().any().any():
    raise ValueError("Missing event identifiers or prediction dates")
if (
    aligned_company.duplicated(["company_id", "prediction_date"]).any()
    or not aligned_company["event_id"].is_unique
):
    raise ValueError("Aligned data must contain one row per company-event")
# Parse all existing event/source dates without changing their observation dates.
date_columns = [
    column
    for column in aligned_company.columns
    if column.endswith("_date") or column.startswith("market_monthly_price_date_")
]
for column in date_columns:
    source = aligned_company[column]
    converted = pd.to_datetime(source, errors="coerce").dt.normalize()
    if (source.notna() & converted.isna()).any():
        raise ValueError(f"Invalid aligned dates in {column}")
    aligned_company[column] = converted
filing_source = company_availability["bankruptcy_date"]
company_availability["bankruptcy_date"] = pd.to_datetime(
    filing_source, errors="coerce"
).dt.normalize()
if (filing_source.notna() & company_availability["bankruptcy_date"].isna()).any():
    raise ValueError("Invalid bankruptcy filing dates")
for column in original_columns:
    if not pd.api.types.is_numeric_dtype(aligned_company[column]):
        raise ValueError(f"Expected an already numeric original field: {column}")
    if not np.isfinite(
        aligned_company.loc[aligned_company[column].notna(), column]
    ).all():
        raise ValueError(f"Nonfinite values in {column}; inspect upstream cleaning")

In [3]:
# Add metadata using a validated many-to-one join; never duplicate event rows.
original_features = aligned_company.merge(
    company_availability,
    on="company_id",
    how="left",
    validate="many_to_one",
    indicator=True,
    sort=False,
)
if not original_features["_merge"].eq("both").all():
    raise ValueError("An aligned company is absent from company_availability.csv")
original_features = original_features.drop(columns="_merge").rename(
    columns={"bloomberg_sector": "sector", "bloomberg_industry": "industry"}
)
for column in ["sector", "industry"]:
    if (
        original_features[column].isna().any()
        or original_features[column].astype(str).str.strip().eq("").any()
    ):
        raise ValueError(f"Missing Bloomberg {column}")

# Use ordinary logs only for nonnegative scale fields; retain signed logs for variables
# whose negative values carry economic meaning.
original_features["log_total_assets"] = np.log1p(
    original_features["total_assets"].clip(lower=0)
)
original_features["log_market_cap"] = np.log1p(
    original_features["market_cap"].clip(lower=0)
)
for column in SIGNED_LOG_FIELDS:
    values = original_features[column]
    original_features[f"signed_log_{column}"] = np.sign(values) * np.log1p(
        np.abs(values)
    )

# Indicators describe original numeric fields only; missing values stay missing.
missing_indicators = pd.DataFrame(
    {
        f"{column}_missing": original_features[column].isna().astype("int8")
        for column in original_columns
    },
    index=original_features.index,
)
original_features = pd.concat([original_features, missing_indicators], axis=1)
front_columns = (
    [
        "company_id",
        "event_id",
        "bankruptcy_date",
        "prediction_date",
        "target",
        "sector",
        "industry",
        "prediction_year",
    ]
    + original_columns
    + missing_indicators.columns.tolist()
)
remaining_columns = [
    column for column in original_features.columns if column not in front_columns
]
original_features = original_features[front_columns + remaining_columns]

missingness_report = pd.DataFrame(
    {
        "field": original_columns,
        "missing_count": [
            int(original_features[column].isna().sum()) for column in original_columns
        ],
        "missing_pct": [
            100 * original_features[column].isna().mean() for column in original_columns
        ],
    }
).sort_values(["missing_pct", "field"], ascending=[False, True], kind="stable")

In [4]:
# Verify row identity, labels, unaltered numeric values, and exact missingness flags.
for column in original_columns:
    flag = original_features[f"{column}_missing"]
expected_filings = original_features["company_id"].map(
    company_availability.set_index("company_id")["bankruptcy_date"]
)
positive = original_features["target"].eq(1)

output_path = PROCESSED / "original_features.csv"
original_features.to_csv(output_path, index=False, date_format="%Y-%m-%d")